# Roosterly: Kaggle GPU evaluation
This notebook clones the private `kaggle-gpu-inference` branch using a read-only token from Kaggle Secrets, selects the local Qwen model, and evaluates labeled examples without posting ticket actions. Keep the notebook private.


In [ ]:
import os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
repo = Path('/kaggle/working/Roosterly')
if not (repo / 'requirements-kaggle.txt').exists():
    token = UserSecretsClient().get_secret('GITHUB_READ_TOKEN')
    askpass = Path('/kaggle/working/github_askpass.sh')
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token;; *) printf %s "$ROOSTERLY_GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    clone_env = os.environ.copy()
    clone_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', ROOSTERLY_GITHUB_TOKEN=token)
    try:
        subprocess.run(['git', 'clone', '--branch', 'kaggle-gpu-inference', '--single-branch', 'https://github.com/himanrawat/Roosterly.git', str(repo)], check=True, env=clone_env)
    finally:
        askpass.unlink(missing_ok=True)
        del token, clone_env
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-kaggle.txt')], check=True)
print('Repository and dependencies ready')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Session options first'
print(torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ('HELPDESK_API_KEY', 'CANDIDATE_EMAIL'):
    os.environ[name] = secrets.get_secret(name)
os.environ['AGENT_PROVIDER'] = 'local'
os.environ['LOCAL_MODEL'] = 'Qwen/Qwen3-8B'
print('Local provider selected; required secrets loaded')


In [ ]:
# Read-only evaluation. This does not send ticket actions.
result = subprocess.run([sys.executable, '-m', 'agent.evaluate', '--label', 'kaggle_qwen3', '--runs', '1', '--force'], cwd=repo, env=os.environ.copy())
print('Evaluation exit code:', result.returncode)


## Posting tickets
Review the evaluation and actual draft replies before running the live command. When ready, run `python -m agent` from the repository directory in a separate cell. Do not reset the existing scored practice run just to test Kaggle.
